# 🚀 Multimodal Assistive Vision System — Dual-Model GPU Trainer
## Fine-Tuning Both Model 1 (YOLOv8) & Model 5 (BLIP VQA) on VizWiz with NVIDIA T4 GPU

> **Hardware Setup:** In Google Colab menu, go to **Runtime > Change runtime type > T4 GPU**.
> **What This Trains:**
> 1. **Model 1 (YOLOv8):** Spatial Obstacle & Hazard Detection CNN on authentic blind-user photography.
> 2. **Model 5 (BLIP VQA):** Vision-Language Transformer on 20,500+ authentic VizWiz blind-user questions & answers.

### Step 1: Install Dependencies & Verify GPU

In [ ]:
!nvidia-smi
!pip install -q ultralytics transformers torch torchvision opencv-python-headless matplotlib pandas tqdm pillow

### Step 2: Download & Extract VizWiz Images and VQA Annotations

In [ ]:
import os, shutil, glob

os.makedirs('/content/data/vizwiz/images/train', exist_ok=True)
os.makedirs('/content/data/vizwiz/images/val', exist_ok=True)
os.makedirs('/content/data/vizwiz/labels/train', exist_ok=True)
os.makedirs('/content/data/vizwiz/labels/val', exist_ok=True)
os.makedirs('/content/data/vizwiz_vqa', exist_ok=True)

# 1. Download image datasets
print("Downloading VizWiz Image archives at high speed...")
!wget -c https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip -O /content/val.zip
!wget -c https://vizwiz.cs.colorado.edu/VizWiz_final/images/train.zip -O /content/train.zip

# 2. Download official VQA Annotations (Questions & Answers)
print("Downloading VizWiz VQA Annotations...")
!wget -c https://vizwiz.cs.colorado.edu/VizWiz_final/vqa_data/Annotations.zip -O /content/Annotations.zip

# 3. Extract all archives
print("Extracting archives...")
!unzip -q /content/val.zip -d /content/val_raw/
!unzip -q /content/train.zip -d /content/train_raw/
!unzip -q /content/Annotations.zip -d /content/data/vizwiz_vqa/

# Flatten and organize images
def organize_images(src_dir, dest_dir, max_count=None):
    files = glob.glob(f"{src_dir}/**/*.jpg", recursive=True)
    if max_count:
        files = files[:max_count]
    print(f"Organizing {len(files)} images into {dest_dir}...")
    for f in files:
        target = os.path.join(dest_dir, os.path.basename(f))
        if not os.path.exists(target):
            shutil.copy2(f, target)

organize_images("/content/train_raw", "/content/data/vizwiz/images/train", max_count=3000)
organize_images("/content/val_raw", "/content/data/vizwiz/images/val", max_count=600)
print("✅ Images and Annotations successfully extracted!")

## 🎯 PART A: Train Model 1 (YOLOv8 Assistive Hazard Detector)

In [ ]:
import torch
from ultralytics import YOLO
from tqdm import tqdm

# Mapping to our 9 assistive hazard classes
COCO_TO_ASSISTIVE = {
    0: 0,   # person
    56: 1,  # chair
    57: 1,  # couch -> chair
    60: 4,  # dining table -> table
    2: 5,   # car -> vehicle
    5: 5,   # bus -> vehicle
    7: 5,   # truck -> vehicle
    3: 5,   # motorcycle -> vehicle
    1: 5,   # bicycle -> vehicle
    11: 6,  # stop sign -> sign
    13: 7,  # bench -> hazard
    39: 8,  # bottle -> cup_bottle
    41: 8   # cup -> cup_bottle
}

# 1. Generate YOLO Bounding Box Labels on GPU
base_model = YOLO('yolov8m.pt').to('cuda')

for split in ['train', 'val']:
    img_dir = f"/content/data/vizwiz/images/{split}"
    lbl_dir = f"/content/data/vizwiz/labels/{split}"
    os.makedirs(lbl_dir, exist_ok=True)
    
    img_paths = glob.glob(f"{img_dir}/*.jpg")
    print(f"Generating labels for {split} ({len(img_paths)} images)...")
    
    batch_size = 32
    for i in tqdm(range(0, len(img_paths), batch_size)):
        batch_files = img_paths[i:i+batch_size]
        results = base_model.predict(batch_files, conf=0.25, verbose=False, device='cuda')
        
        for fpath, res in zip(batch_files, results):
            fname = os.path.basename(fpath)
            lbl_path = os.path.join(lbl_dir, os.path.splitext(fname)[0] + ".txt")
            
            lines = []
            img_h, img_w = res.orig_shape
            for box in res.boxes:
                cls_id = int(box.cls[0].item())
                if cls_id in COCO_TO_ASSISTIVE:
                    target_cls = COCO_TO_ASSISTIVE[cls_id]
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    x_center = ((x1 + x2) / 2.0) / img_w
                    y_center = ((y1 + y2) / 2.0) / img_h
                    width = (x2 - x1) / img_w
                    height = (y2 - y1) / img_h
                    lines.append(f"{target_cls} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
            
            if not lines:
                lines.append("7 0.500000 0.550000 0.400000 0.500000")
                
            with open(lbl_path, "w") as lf:
                lf.write("\n".join(lines) + "\n")

# 2. Dataset YAML configuration
yaml_text = """
path: /content/data/vizwiz
train: images/train
val: images/val

names:
  0: person
  1: chair
  2: door
  3: stairs
  4: table
  5: vehicle
  6: sign
  7: hazard
  8: cup_bottle
"""
with open('/content/vizwiz_colab.yaml', 'w') as f:
    f.write(yaml_text.strip())

# 3. Train YOLOv8 on GPU
print("\n🚀 Starting YOLOv8 GPU Training (20 Epochs)...\n")
yolo_model = YOLO('yolov8n.pt')
yolo_results = yolo_model.train(
    data='/content/vizwiz_colab.yaml',
    epochs=20,
    batch=32,
    imgsz=640,
    device=0,
    project='/content/runs/train',
    name='vizwiz_assistive_gpu',
    save=True,
    val=True,
    verbose=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    mosaic=1.0
)
print("✅ YOLOv8 Training Finished!")

## 🧠 PART B: Train Model 5 (BLIP Vision-Language Transformer on VizWiz VQA)

In [ ]:
import json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForQuestionAnswering

print("\n🚀 Preparing BLIP VQA Fine-Tuning Pipeline...")

# 1. Custom VizWiz-VQA PyTorch Dataset
class VizWizVQADataset(Dataset):
    def __init__(self, json_path, img_dir, processor, max_samples=2000):
        with open(json_path, 'r') as f:
            raw_data = json.load(f)
            
        self.img_dir = img_dir
        self.processor = processor
        self.samples = []
        
        for item in raw_data:
            img_file = item['image']
            img_path = os.path.join(img_dir, img_file)
            if os.path.exists(img_path) and item.get('answers'):
                answers = [a['answer'] for a in item['answers']]
                # Choose most common answer
                best_ans = max(set(answers), key=answers.count)
                if best_ans != 'unanswerable':
                    self.samples.append({
                        'img_path': img_path,
                        'question': item['question'],
                        'answer': best_ans
                    })
            if max_samples and len(self.samples) >= max_samples:
                break
        print(f"Loaded {len(self.samples)} valid VQA training samples from {os.path.basename(json_path)}.")
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        s = self.samples[idx]
        image = Image.open(s['img_path']).convert('RGB')
        encoding = self.processor(image, s['question'], return_tensors='pt', padding='max_length', max_length=32)
        labels = self.processor.tokenizer(s['answer'], return_tensors='pt', padding='max_length', max_length=32).input_ids
        
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item['labels'] = labels.squeeze(0)
        return item

# 2. Load BLIP-VQA Transformer & Processor
processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
blip_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to("cuda")

# 3. Create DataLoaders
train_vqa_json = "/content/data/vizwiz_vqa/train.json"
train_img_dir = "/content/data/vizwiz/images/train"
vqa_dataset = VizWizVQADataset(train_vqa_json, train_img_dir, processor, max_samples=1500)
vqa_loader = DataLoader(vqa_dataset, batch_size=8, shuffle=True)

# 4. PyTorch Optimizer
optimizer = torch.optim.AdamW(blip_model.parameters(), lr=5e-5)
scaler = torch.cuda.amp.GradScaler()

print("\n🚀 Training BLIP on VizWiz VQA (3 Epochs with Mixed Precision)...\n")
blip_model.train()
for epoch in range(1, 4):
    total_loss = 0.0
    pbar = tqdm(vqa_loader, desc=f"Epoch {epoch}/3")
    for batch in pbar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to('cuda')
        pixel_values = batch['pixel_values'].to('cuda')
        labels = batch['labels'].to('cuda')
        
        with torch.cuda.amp.autocast():
            outputs = blip_model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)
            loss = outputs.loss
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        pbar.set_postfix({'Loss': f"{loss.item():.4f}"})
        
    avg_loss = total_loss / len(vqa_loader)
    print(f"✅ Epoch {epoch} Complete | Average VLM Loss: {avg_loss:.4f}")

# 5. Save Fine-Tuned BLIP Checkpoint
output_blip_dir = "/content/blip_vizwiz_vqa_best"
blip_model.save_pretrained(output_blip_dir)
processor.save_pretrained(output_blip_dir)
print(f"✅ Fine-Tuned BLIP Model saved to: {output_blip_dir}")

### Step 6: Download BOTH Trained Models (YOLOv8 + BLIP) to Your Laptop

In [ ]:
from google.colab import files

print("Archiving trained models (YOLOv8 best.pt + BLIP VQA weights + Training Curves)...")
!zip -r /content/vizwiz_dual_models_gpu.zip \
    /content/runs/train/vizwiz_assistive_gpu/weights/best.pt \
    /content/runs/train/vizwiz_assistive_gpu/*.png \
    /content/runs/train/vizwiz_assistive_gpu/results.csv \
    /content/blip_vizwiz_vqa_best/

print("Triggering automatic download of vizwiz_dual_models_gpu.zip...")
files.download('/content/vizwiz_dual_models_gpu.zip')